# E51 --- o limiar que se recalibra, e o falso que se assina

**A tentativa.** O capitulo mediu que o limiar calibrado na escala do mundo antigo fica pequeno
demais quando o mundo dobra: a janela despenca e nao para mais de cortar. Esta travessia mede o
conserto pelo eixo do tempo --- o limiar que se recalibra dia a dia pela propria cadencia --- e a
garantia pelo eixo da bateria --- a selecao e-BH, que poe a fracao de falsos no lugar da contagem.

**O que se mede.**

1. a cadencia de encolhimentos por ano, no mundo que nunca muda e no mundo que dobra, para o limiar
   fixo e para o limiar recalibrado em tres ganhos declarados;
2. o preco dessa obediencia: o tamanho da janela depois da mudanca;
3. o caminho do limiar, dia a dia, nos dois mundos;
4. a bateria e-BH: quantos falsos ela compra no mundo parado, e em quantos dias ela compra o corte
   verdadeiro depois do degrau.

**Convencoes** (AGENTS.md paragrafos 7 e 9): um experimento por caderno, parametros no topo
marcados com "brinque com", algoritmo em frevolab, resultado em lab/resultados/E51_recalibra.json,
figuras em .pdf e .png.

In [1]:
# <- brinque com: DIAS, QUANDO, FATOR, MUNDOS, MINIMA, MAXIMA, ORCAMENTO, GAMAS, TAXA_EBH, DIAS_UTEIS, SEMENTE
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

import frevolab
from frevolab import adaptativa, graficos, mudanca, multiplicidade

DIAS = 14000                   # o mundo longo do capitulo
QUANDO = 4000                  # o dia em que a escala dobra
FATOR = 2.0
MUNDOS = 12                    # os mundos do mesmo sorteio
MINIMA = 21
MAXIMA = 1008
ORCAMENTO = 1.0                # um falso por ano, a unidade do capitulo do alarme
GAMAS = (0.02, 0.05, 0.2)      # os ganhos declarados da recalibragem
TAXA_EBH = 0.05                # a mesma taxa que o corte do comeco do livro promessa
DIAS_UTEIS = 252
SEMENTE = 118                  # E40..E53 usam 107..120

SIGMA = mudanca.SIGMA_PADRAO
NIVEL_BASE = SIGMA * np.sqrt(2.0 / np.pi)   # a convencao do capitulo: o nivel e o |retorno| medio
rng = np.random.default_rng(SEMENTE)
parado = [np.abs(mudanca.estavel(DIAS, rng, SIGMA)) for _ in range(MUNDOS)]
dobra = [np.abs(mudanca.degrau(DIAS, rng, SIGMA, fator=FATOR, quando=QUANDO))
         for _ in range(MUNDOS)]
verdade_parado = np.full(DIAS, NIVEL_BASE)
verdade_dobra = np.concatenate([np.full(QUANDO, NIVEL_BASE),
                               np.full(DIAS - QUANDO, NIVEL_BASE * FATOR)])
limiar_fixo = adaptativa.limiar_do_orcamento(parado[0], ORCAMENTO, MINIMA, MAXIMA, DIAS_UTEIS)

print("frevolab %s | %d mundos de %d dias | orcamento %.1f falso por ano | limiar fixo %.6f"
      % (frevolab.VERSAO, MUNDOS, DIAS, ORCAMENTO, limiar_fixo))

frevolab 0.1.0 | 12 mundos de 14000 dias | orcamento 1.0 falso por ano | limiar fixo 0.001265


## A cadencia: o orcamento, e quem o obedece

A cadencia de encolhimentos por ano e a unidade do orcamento. No mundo em que o limiar foi
calibrado ela bate; no mundo que dobra, o limiar fixo multiplica os cortes --- e o recalibrado os
traz de volta, ao preco de segurar a janela.

In [2]:
braços = {}
braços["fixo no parado"] = [adaptativa.nivel(s, limiar_fixo, MINIMA, MAXIMA) for s in parado]
braços["fixo na dobra"] = [adaptativa.nivel(s, limiar_fixo, MINIMA, MAXIMA) for s in dobra]
for gama in GAMAS:
    braços["gama %.2f" % gama] = [adaptativa.nivel_recalibrado(s, ORCAMENTO, gama, MINIMA, MAXIMA,
                                                               DIAS_UTEIS) for s in dobra]
print("%-16s %10s %12s %10s" % ("braco", "cortes", "por ano", "janela"))
medidas = {}
for nome, saidas in braços.items():
    cortes = float(np.mean([len(s["encolhimentos"]) for s in saidas]))
    cad = float(np.mean([adaptativa.falsos_por_ano(s["encolhimentos"], DIAS, DIAS_UTEIS)
                          for s in saidas]))
    jan = float(np.median([np.median(s["tamanhos"][QUANDO:]) for s in saidas]))
    medidas[nome] = {"cortes": cortes, "cadencia": cad, "janela": jan}
    print("%-16s %10.0f %12.2f %10.0f" % (nome, cortes, cad, jan))

braco                cortes      por ano     janela
fixo no parado          116         2.08       1008
fixo na dobra           538         9.68         32
gama 0.02                58         1.04       1008
gama 0.05                66         1.18       1008
gama 0.20                75         1.35        772


In [3]:
# Figura 1: a cadencia contra o orcamento, e o preco na janela.
fig, eixos = plt.subplots(1, 2, figsize=(10.4, 4.0))
nomes = list(braços)
eixos[0].bar(range(len(nomes)), [medidas[n]["cadencia"] for n in nomes], color="#1f4e79")
eixos[0].axhline(ORCAMENTO, color="#b03a2e", ls="--", lw=1.5,
                 label="o orcamento: %.1f por ano" % ORCAMENTO)
eixos[0].set_xticks(range(len(nomes)))
eixos[0].set_xticklabels(nomes, rotation=25, ha="right", fontsize=8)
eixos[0].set_ylabel("cortes por ano")
eixos[0].set_title("a cadencia, e o orcamento", fontsize=10)
eixos[0].legend(fontsize=8)
eixos[1].bar(range(len(nomes)), [medidas[n]["janela"] for n in nomes], color="#c78f2c")
eixos[1].axhline(MAXIMA, color="0.4", ls=":", lw=1.2, label="a janela maxima: %d" % MAXIMA)
eixos[1].set_xticks(range(len(nomes)))
eixos[1].set_xticklabels(nomes, rotation=25, ha="right", fontsize=8)
eixos[1].set_ylabel("janela mediana depois da mudanca (dias)")
eixos[1].set_title("o preco: onde a janela aterrissa", fontsize=10)
eixos[1].legend(fontsize=8)
fig.tight_layout()
graficos.salvar(fig, "E51_recalibra", 1)
plt.close(fig)
print("figura E51_recalibra_1 salva")

figura E51_recalibra_1 salva


## O caminho do limiar

O limiar recalibrado sobe quando os cortes estouram o orcamento e derrete quando o ano corre sem
cortar. No mundo parado ele anda pouco; no mundo que dobra ele sobe e assenta noutro patamar.

In [4]:
GAMA_FIGURA = GAMAS[1]
caminho_dobra = [adaptativa.nivel_recalibrado(s, ORCAMENTO, GAMA_FIGURA, MINIMA, MAXIMA, DIAS_UTEIS)
                 for s in dobra]
caminho_parado = [adaptativa.nivel_recalibrado(s, ORCAMENTO, GAMA_FIGURA, MINIMA, MAXIMA, DIAS_UTEIS)
                  for s in parado]
limiar_dobra = np.median(np.array([[l for _t, l, _d in s["limiares"]] for s in caminho_dobra]), axis=0)
limiar_parado = np.median(np.array([[l for _t, l, _d in s["limiares"]] for s in caminho_parado]), axis=0)
print("limiar mediano: no parado comeca %.6f e termina %.6f | na dobra comeca %.6f e termina %.6f"
      % (limiar_parado[0], limiar_parado[-1], limiar_dobra[0], limiar_dobra[-1]))

limiar mediano: no parado comeca 0.001412 e termina 0.001315 | na dobra comeca 0.002416 e termina 0.002623


In [5]:
# Figura 2: o caminho do limiar nos dois mundos.
fig, eixos = plt.subplots(2, 1, figsize=(9.0, 5.2), sharex=True)
eixos[0].plot(limiar_dobra, color="#1f4e79", lw=1.4, label="o limiar recalibrado")
eixos[0].axhline(limiar_fixo, color="#b03a2e", ls="--", lw=1.2, label="o limiar fixo (calibrado no parado)")
eixos[0].axvline(QUANDO, color="0.5", ls=":", lw=1.0)
eixos[0].set_ylabel("limiar")
eixos[0].set_title("no mundo que dobra (mudanca marcada)", fontsize=10)
eixos[0].legend(fontsize=8)
eixos[1].plot(limiar_parado, color="#1f4e79", lw=1.4, label="o limiar recalibrado")
eixos[1].axhline(limiar_fixo, color="#b03a2e", ls="--", lw=1.2)
eixos[1].set_xlabel("dias")
eixos[1].set_ylabel("limiar")
eixos[1].set_title("no mundo que nunca muda", fontsize=10)
eixos[1].legend(fontsize=8)
fig.tight_layout()
graficos.salvar(fig, "E51_recalibra", 2)
plt.close(fig)
print("figura E51_recalibra_2 salva")

figura E51_recalibra_2 salva


## A bateria: o falso que se assina

Na bateria de dias, a selecao e-BH compra as descobertas com a fracao de falsos assinada --- e ela
controla essa fracao sob dependencia arbitraria entre os dias, porque a media de e-valores e
e-valor. No mundo parado nao ha o que descobrir; no mundo que dobra, a descoberta custa uma
latencia.

In [6]:
e_parado = adaptativa.e_das_metades(parado[0], MINIMA, MAXIMA)["e"]
e_dobra = adaptativa.e_das_metades(dobra[0], MINIMA, MAXIMA)["e"]
selecao_parado = multiplicidade.selecao_ebh(e_parado[2 * MINIMA:], TAXA_EBH)
selecao_dobra = multiplicidade.selecao_ebh(e_dobra[2 * MINIMA:], TAXA_EBH)
indices_dobra = selecao_dobra["indices"] + 2 * MINIMA
primeiro = int(indices_dobra.min()) if indices_dobra.size else -1
indices_parado = selecao_parado["indices"] + 2 * MINIMA
primeiro_parado = int(indices_parado.min()) if indices_parado.size else -1
cortes_fixos_parado = float(np.mean([len(s["encolhimentos"]) for s in braços["fixo no parado"]]))
print("mundo parado: e-BH seleciona %d de %d dias | o limiar fixo corta %.0f vezes (%.2f por ano)"
      % (selecao_parado["quantidade"], selecao_parado["comparacoes"], cortes_fixos_parado,
         medidas["fixo no parado"]["cadencia"]))
print("mundo que dobra: e-BH seleciona %d dias | primeiro em %s | latencia %s dias"
      % (selecao_dobra["quantidade"], primeiro,
         (primeiro - QUANDO) if primeiro > 0 else "sem corte"))

mundo parado: e-BH seleciona 0 de 13958 dias | o limiar fixo corta 116 vezes (2.08 por ano)
mundo que dobra: e-BH seleciona 732 dias | primeiro em 4085 | latencia 85 dias


In [7]:
# Figura 3: a bateria --- falsos no parado e latencia na dobra.
fig, eixos = plt.subplots(1, 2, figsize=(9.6, 3.8))
eixos[0].bar(["limiar fixo (por ano)", "e-BH (por ano)"],
             [medidas["fixo no parado"]["cadencia"],
              252.0 * selecao_parado["quantidade"] / float(selecao_parado["comparacoes"])],
             color=["#b03a2e", "#1f4e79"])
eixos[0].set_ylabel("por ano")
eixos[0].set_title("no mundo que nao muda: quem compra falso", fontsize=10)
latencia_fixo = float(np.median([sorted(d for d, _a, _b in t["encolhimentos"] if d > QUANDO)[0]
                            for t in braços["fixo na dobra"]
                            if any(d > QUANDO for d, _a, _b in t["encolhimentos"])])) - QUANDO
eixos[1].bar(["limiar fixo", "e-BH"],
             [latencia_fixo, (primeiro - QUANDO) if primeiro > 0 else 0.0],
             color=["#b03a2e", "#1f4e79"])
eixos[1].set_ylabel("dias ate o primeiro corte depois da mudanca")
eixos[1].set_title("no mundo que dobra: a latencia", fontsize=10)
fig.tight_layout()
graficos.salvar(fig, "E51_recalibra", 3)
plt.close(fig)
print("figura E51_recalibra_3 salva")

figura E51_recalibra_3 salva


## Leitura visual das figuras

**Declarada contra os .png depois da execucao** (AGENTS.md paragrafo 9).

O que as legendas do capitulo afirmam, e a leitura tem de conferir nos .png:

1. **Figura 1**: no painel esquerdo a barra do limiar fixo na dobra muito acima da linha do
   orcamento, e as barras dos ganhos em volta da linha; no direito a janela do fixo na dobra muito
   abaixo das outras.
2. **Figura 2**: no painel de cima o limiar sobe depois da mudanca e assenta; no de baixo ele anda
   pouco.
3. **Figura 3**: no painel esquerdo a barra do limiar fixo acima da do e-BH; no direito a barra da
   latencia do e-BH.

In [8]:
# O resultado: um objeto por grandeza, em portugues, para o livro citar por comando.
chave_gama_meio = "gama %.2f" % GAMAS[1]
resultado = {
    "recalibra_mundos": MUNDOS,
    "recalibra_dias": DIAS,
    "recalibra_orcamento": ORCAMENTO,
    "recalibra_gamas": len(GAMAS),
    "recalibra_limiar_fixo": round(limiar_fixo, 6),
    "recalibra_cadencia_parado": round(medidas["fixo no parado"]["cadencia"], 3),
    "recalibra_cadencia_dobra": round(medidas["fixo na dobra"]["cadencia"], 3),
    "recalibra_janela_parado": round(medidas["fixo no parado"]["janela"], 1),
    "recalibra_janela_dobra": round(medidas["fixo na dobra"]["janela"], 1),
    "recalibra_cortes_dobra": round(medidas["fixo na dobra"]["cortes"], 1),
    "recalibra_cadencia_gama_baixo": round(medidas["gama %.2f" % GAMAS[0]]["cadencia"], 3),
    "recalibra_cadencia_gama_meio": round(medidas[chave_gama_meio]["cadencia"], 3),
    "recalibra_cadencia_gama_alto": round(medidas["gama %.2f" % GAMAS[-1]]["cadencia"], 3),
    "recalibra_janela_gama_meio": round(medidas[chave_gama_meio]["janela"], 1),
    "recalibra_razao_cadencias": round(medidas["fixo na dobra"]["cadencia"]
                                       / medidas[chave_gama_meio]["cadencia"], 2),
    "recalibra_limiar_parado_inicio": round(float(limiar_parado[0]), 6),
    "recalibra_limiar_parado_fim": round(float(limiar_parado[-1]), 6),
    "recalibra_limiar_dobra_inicio": round(float(limiar_dobra[0]), 6),
    "recalibra_limiar_dobra_fim": round(float(limiar_dobra[-1]), 6),
    "recalibra_ebh_dias_parado": int(selecao_parado["quantidade"]),
    "recalibra_ebh_comparacoes_parado": int(selecao_parado["comparacoes"]),
    "recalibra_ebh_dias_dobra": int(selecao_dobra["quantidade"]),
    "recalibra_ebh_latencia": int(primeiro - QUANDO) if primeiro > 0 else -1,
    "recalibra_latencia_fixo": int(latencia_fixo),
    "recalibra_ebh_primeiro_parado": int(primeiro_parado) if primeiro_parado > 0 else -1,
    "recalibra_taxa_ebh_pct": round(100 * TAXA_EBH, 1),
    "recalibra_e_media_parado": round(float(np.mean(e_parado[2 * MINIMA:])), 4),
}
caminho = Path("lab/resultados/E51_recalibra.json")
caminho.parent.mkdir(parents=True, exist_ok=True)
caminho.write_text(json.dumps(resultado, ensure_ascii=False, indent=1), encoding="utf-8")
print(json.dumps(resultado, ensure_ascii=False, indent=1))

{
 "recalibra_mundos": 12,
 "recalibra_dias": 14000,
 "recalibra_orcamento": 1.0,
 "recalibra_gamas": 3,
 "recalibra_limiar_fixo": 0.001265,
 "recalibra_cadencia_parado": 2.079,
 "recalibra_cadencia_dobra": 9.677,
 "recalibra_janela_parado": 1008.0,
 "recalibra_janela_dobra": 32.0,
 "recalibra_cortes_dobra": 537.6,
 "recalibra_cadencia_gama_baixo": 1.041,
 "recalibra_cadencia_gama_meio": 1.182,
 "recalibra_cadencia_gama_alto": 1.353,
 "recalibra_janela_gama_meio": 1008.0,
 "recalibra_razao_cadencias": 8.19,
 "recalibra_limiar_parado_inicio": 0.001412,
 "recalibra_limiar_parado_fim": 0.001315,
 "recalibra_limiar_dobra_inicio": 0.002416,
 "recalibra_limiar_dobra_fim": 0.002623,
 "recalibra_ebh_dias_parado": 0,
 "recalibra_ebh_comparacoes_parado": 13958,
 "recalibra_ebh_dias_dobra": 732,
 "recalibra_ebh_latencia": 85,
 "recalibra_latencia_fixo": 75,
 "recalibra_ebh_primeiro_parado": -1,
 "recalibra_taxa_ebh_pct": 5.0,
 "recalibra_e_media_parado": 0.9672
}
